# 6. Resource benchmark

The cost side of the comparison: parameters, size on disk, multiply-accumulates, latency and peak
memory of all four models, measured by the same code, in one session, on one machine. The models
are built fresh, so no training or checkpoint is needed.

Latency is the only number here that depends on the machine. It is measured on one CPU thread, as
a small device without a GPU would run the model, so run this notebook on a computer that is
otherwise idle, plugged in if it is a laptop.

**Run time:** about a minute.

**Writes:** `results/benchmark.json`.

## Setup

Folders, the random seed, the device, and small helpers for files and figures. Every notebook in
this folder starts with the same setup.

In [1]:
import json
import os
import platform
import random
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torchvision

# Everything is read from and written to folders next to this notebook.
DATA_DIR = Path("data")                # the downloaded dataset and a decoded-pixel cache
RESULTS_DIR = Path("results")          # metrics, figures and tables
CHECKPOINT_DIR = Path("checkpoints")   # trained weights

SEED = 42            # fixes the split, the initial weights, augmentation and batch order
NUM_CLASSES = 10     # EuroSAT land-cover classes
INPUT_SHAPE = (3, 64, 64)

# cuBLAS needs this workspace setting for repeatable matrix products on a GPU.
CUBLAS_WORKSPACE = ":4096:8"


def seed_everything(seed, deterministic=True):
    """Seed Python, NumPy and PyTorch, so a run gives the same numbers every time.

    With ``deterministic=True``, cuDNN uses deterministic algorithms and stops
    auto-tuning, and PyTorch prefers deterministic implementations. This costs a
    little speed. Without it, two runs of the same experiment can differ by a few
    tenths of a percent, the same size as some differences between the models.
    A few GPU operations have no deterministic version; PyTorch then warns
    instead of stopping.
    """
    if deterministic:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", CUBLAS_WORKSPACE)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # seeds the CPU and every GPU
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def cpu_name():
    """The processor's model name, or an empty string if the operating system won't say."""
    name = ""
    try:
        if sys.platform == "win32":
            import winreg

            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0"
            )
            name = str(winreg.QueryValueEx(key, "ProcessorNameString")[0])
        elif sys.platform.startswith("linux"):
            with open("/proc/cpuinfo", encoding="utf-8") as handle:
                for line in handle:
                    if line.lower().startswith("model name"):
                        name = line.split(":", 1)[1]
                        break
        elif sys.platform == "darwin":
            name = subprocess.run(
                ["sysctl", "-n", "machdep.cpu.brand_string"],
                capture_output=True, text=True, timeout=5, check=False,
            ).stdout
    except Exception:
        name = ""
    name = name or platform.processor() or ""
    name = re.sub(r"\((R|TM|C)\)", "", name, flags=re.IGNORECASE)  # trademark signs
    return " ".join(name.split())


def describe_device(device):
    """A readable name for a device, such as ``cuda:0 (Tesla T4)``.

    Every timing is recorded together with this name: a time means little
    without the hardware it was measured on.
    """
    device = torch.device(device)
    if device.type == "cuda":
        index = device.index if device.index is not None else torch.cuda.current_device()
        return f"cuda:{index} ({torch.cuda.get_device_name(index)})"
    name = cpu_name()
    return f"cpu ({name})" if name else "cpu"


#: Model name -> function building it; filled in by the model definitions below.
MODEL_BUILDERS = {}


def build_model(name, num_classes=NUM_CLASSES):
    """Seed PyTorch, then build ``name``: the same seed always gives the same initial weights.

    The three optimizer runs rely on this to start from identical weights, so
    that their comparison measures the optimizer and not the initialisation.
    """
    torch.manual_seed(SEED)
    return MODEL_BUILDERS[name](num_classes)


def count_parameters(model, trainable_only=True):
    """Number of parameters; frozen ones are left out when ``trainable_only``."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def model_size_kb(model):
    """Size of the saved weights (``state_dict``) in KB, where 1 KB = 1,024 bytes.

    Saving the weights, as a device would store them, rather than counting 4
    bytes per parameter, also counts buffers such as batch-norm statistics.
    """
    import io

    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1024


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python       {platform.python_version()} ({platform.system()})")
print(f"PyTorch      {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"NumPy        {np.__version__}")
print(f"Device       {describe_device(DEVICE)}")

Python       3.11.3 (Windows)
PyTorch      2.14.0+cu126
torchvision  0.29.0+cu126
NumPy        2.4.6
Device       cuda:0 (NVIDIA GeForce RTX 3050 Laptop GPU)


Figures are shown inline at a small size and saved at print quality. Each model and each optimizer
keeps one colour and one name in every figure and table.

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 100,            # inline figures: small, so the notebook stays light
    "savefig.dpi": 200,           # saved figures: print quality
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 1.8,
})

MODEL_LABELS = {
    "model_a": "Model A",
    "model_b": "Model B",
    "mobilenet_v2": "MobileNetV2",
    "squeezenet1_1": "SqueezeNet 1.1",
}
MODEL_COLORS = {
    "model_a": "#4C72B0",
    "model_b": "#DD8452",
    "mobilenet_v2": "#55A868",
    "squeezenet1_1": "#C44E52",
}
OPTIMIZER_LABELS = {"sgd": "SGD", "sgd_momentum": "SGD + momentum", "adam": "Adam"}
OPTIMIZER_COLORS = {"sgd": "#8172B3", "sgd_momentum": "#937860", "adam": "#DA8BC3"}


def save_figure(fig, path):
    """Save a figure as a PNG, creating its folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")

Results are saved as JSON and Markdown, so the tables can be rebuilt without retraining.

In [3]:
def write_json(path, payload):
    """Write ``payload`` as indented JSON, creating the folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8", newline="\n")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


def format_cell(value, spec=None):
    """One table cell: thousands separators for whole numbers, ``spec`` for decimals."""
    if value is None:
        return "-"
    if isinstance(value, bool):
        return str(value)
    if spec is not None and isinstance(value, (int, float)):
        return format(value, spec)
    if isinstance(value, int):
        return f"{value:,}"
    if isinstance(value, float):
        return f"{value:,.1f}" if abs(value) >= 1000 else f"{value:.4g}"
    return str(value).replace("|", "\\|")


def markdown_table(headers, rows, caption=None, formats=None):
    """A Markdown table. Number columns are right-aligned; ``formats`` fixes their decimals."""
    formats = formats or [None] * len(headers)

    def numeric(column):
        cells = [row[column] for row in rows if row[column] is not None]
        return bool(cells) and all(
            isinstance(cell, (int, float)) and not isinstance(cell, bool) for cell in cells
        )

    lines = [f"*{caption}*", ""] if caption else []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("|" + "|".join("---:" if numeric(c) else "---" for c in range(len(headers))) + "|")
    for row in rows:
        lines.append("| " + " | ".join(format_cell(v, s) for v, s in zip(row, formats)) + " |")
    return "\n".join(lines) + "\n"

## The custom models

**Model A** is a standard CNN: three blocks of 3×3 convolution, batch normalisation, ReLU and 2×2 max
pooling, then a fully connected head on the flattened feature maps. It is not limited in size; it is
the reference point that shows what Model B saves and what it costs.

**Model B** stays under 100,000 trainable parameters through two separate mechanisms:

1. **Depthwise-separable convolutions.** A standard 3×3 convolution from `C_in` to `C_out` channels
   has `9 × C_in × C_out` weights. A depthwise 3×3 convolution (one filter per input channel,
   `9 × C_in`) followed by a pointwise 1×1 convolution (`C_in × C_out`) has `9 × C_in + C_in × C_out`,
   about `1/C_out + 1/9` of the standard cost in both parameters and multiply-accumulates.
2. **Global average pooling instead of flattening.** Model A's flatten into a 128-unit layer alone
   holds 8,192 × 128 weights; Model B averages each feature map to one number, so its classifier
   holds only 320 × 10 weights.

Model B uses ReLU6, a single `min(max(x, 0), 6)`: no exponential and no lookup table, so it costs
almost nothing on a microcontroller, and its bounded output range keeps 8-bit quantisation of the
activations well scaled. Convolutions followed by batch normalisation have no bias, because batch
normalisation re-centres the output straight away.

Both models return raw scores (logits); the softmax is part of the cross-entropy loss.

In [4]:
from torch import nn

MODEL_A_SETTINGS = {
    "blocks": [  # each block: convolution, batch norm, ReLU, then 2x2 max pooling
        {"out_channels": 32, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
        {"out_channels": 64, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
        {"out_channels": 128, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
    ],
    "head": {"global_pool": False, "fc_units": [128], "dropout": 0.3},
    "activation": "relu",
    "batch_norm": True,
}

MODEL_B_SETTINGS = {
    "stem": {"out_channels": 16, "kernel_size": 3, "stride": 1, "padding": 1},
    "blocks": [  # each block: depthwise-separable convolution, then optional 2x2 max pooling
        {"out_channels": 32, "stride": 1, "pool": 2},
        {"out_channels": 64, "stride": 1, "pool": 2},
        {"out_channels": 160, "stride": 1, "pool": 2},
        {"out_channels": 320, "stride": 1},
    ],
    "head": {"global_pool": True, "fc_units": [], "dropout": 0.2},
    "activation": "relu6",
    "batch_norm": True,
}

MODEL_B_PARAMETER_BUDGET = 100_000


def get_activation(name):
    """The activation layer for a name in the settings."""
    activations = {"relu": nn.ReLU, "relu6": nn.ReLU6}
    return activations[name]()


def conv_bn_act(in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                activation="relu", batch_norm=True):
    """Convolution, batch normalisation, activation. ``k*k*in*out`` weights."""
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                        padding=padding, bias=not batch_norm)]
    if batch_norm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(get_activation(activation))
    return nn.Sequential(*layers)


def depthwise_separable_conv(in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                             activation="relu6", batch_norm=True):
    """Depthwise k x k convolution, then pointwise 1x1 convolution, each with batch norm and activation.

    ``groups=in_channels`` gives every input channel its own filter, so channels
    are only mixed by the 1x1 convolution. Cost: ``k*k*in + in*out`` weights,
    against ``k*k*in*out`` for a standard convolution.
    """
    layers = [nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding,
                        groups=in_channels, bias=not batch_norm)]                    # depthwise
    if batch_norm:
        layers.append(nn.BatchNorm2d(in_channels))
    layers.append(get_activation(activation))
    layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=not batch_norm))  # pointwise
    if batch_norm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(get_activation(activation))
    return nn.Sequential(*layers)


def build_head(layers, settings, input_shape, num_classes):
    """Add the classifier to a list of feature layers and return the whole model.

    The number of features entering the classifier is found by passing one
    blank image through the feature layers.
    """
    head = settings["head"]
    if head["global_pool"]:
        layers.append(nn.AdaptiveAvgPool2d(1))
    layers.append(nn.Flatten())

    features = nn.Sequential(*layers)
    features.eval()
    with torch.no_grad():
        in_features = features(torch.zeros(1, *input_shape)).shape[1]
    features.train()

    for units in head["fc_units"]:  # hidden layers: linear, activation, dropout
        layers.append(nn.Linear(in_features, units))
        layers.append(get_activation(settings["activation"]))
        layers.append(nn.Dropout(head["dropout"]))
        in_features = units
    if not head["fc_units"]:
        layers.append(nn.Dropout(head["dropout"]))  # no hidden layer: dropout before the output layer
    layers.append(nn.Linear(in_features, num_classes))  # raw scores, no softmax

    model = nn.Sequential(*layers)
    model.num_classes = num_classes
    return model


def build_model_a(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, settings=MODEL_A_SETTINGS):
    """Model A: [convolution, batch norm, ReLU, max pool] x 3, then a fully connected head."""
    layers = []
    channels = input_shape[0]
    for block in settings["blocks"]:
        layers.append(conv_bn_act(channels, block["out_channels"], block["kernel_size"],
                                  block["stride"], block["padding"],
                                  settings["activation"], settings["batch_norm"]))
        if block.get("pool"):
            layers.append(nn.MaxPool2d(block["pool"]))
        channels = block["out_channels"]
    return build_head(layers, settings, input_shape, num_classes)


def build_model_b(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, settings=MODEL_B_SETTINGS):
    """Model B: a standard 3x3 stem, depthwise-separable blocks, global average pooling."""
    stem = settings["stem"]
    layers = [conv_bn_act(input_shape[0], stem["out_channels"], stem["kernel_size"],
                          stem["stride"], stem["padding"],
                          settings["activation"], settings["batch_norm"])]
    channels = stem["out_channels"]
    for block in settings["blocks"]:
        layers.append(depthwise_separable_conv(channels, block["out_channels"],
                                               stride=block["stride"],
                                               activation=settings["activation"],
                                               batch_norm=settings["batch_norm"]))
        if block.get("pool"):
            layers.append(nn.MaxPool2d(block["pool"]))
        channels = block["out_channels"]
    return build_head(layers, settings, input_shape, num_classes)


MODEL_BUILDERS["model_a"] = build_model_a
MODEL_BUILDERS["model_b"] = build_model_b

## The pretrained models

MobileNetV2 (Sandler et al., 2018) and SqueezeNet 1.1 (Iandola et al., 2016) come with weights
trained on ImageNet. They reach efficiency in two different ways: MobileNetV2 uses inverted
residual blocks built from depthwise-separable convolutions, while SqueezeNet's fire modules first
"squeeze" the channels with 1×1 convolutions, then expand them again.

Fine-tuning strategy, the same for both:

- the ImageNet classifier is replaced by a new 10-class layer, initialised with small random weights;
- only the new classifier and the last few feature blocks are trained (3 blocks of MobileNetV2,
  2 fire modules of SqueezeNet); the earlier blocks keep their ImageNet features frozen;
- the trained feature blocks learn at a tenth of the classifier's learning rate, so the pretrained
  features are adjusted gently rather than overwritten.

Both models are fed the same 64×64 images as the custom models, not upsampled to their native
224×224. This keeps the cost comparison honest, at the resolution the sensor actually produces,
but it handicaps them: MobileNetV2's total stride of 32 leaves a 2×2 feature map. Their inputs are
normalised with the ImageNet statistics their weights were trained with.

In [5]:
from torch import nn
from torchvision.models import (
    MobileNet_V2_Weights,
    SqueezeNet1_1_Weights,
    mobilenet_v2,
    squeezenet1_1,
)

# Keep the downloaded ImageNet weights inside this folder, with the dataset.
torch.hub.set_dir(str(DATA_DIR / "torch_hub"))

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

PRETRAINED_SETTINGS = {
    "mobilenet_v2": {"unfreeze_last_n": 3, "dropout": 0.2, "backbone_lr_scale": 0.1},
    "squeezenet1_1": {"unfreeze_last_n": 2, "dropout": 0.5, "backbone_lr_scale": 0.1},
}


def replace_classifier(model, num_classes, dropout):
    """Swap the 1000-class ImageNet output layer for a ``num_classes`` one.

    MobileNetV2 ends in a linear layer; SqueezeNet in a 1x1 convolution followed
    by global average pooling. The new layer starts from small random weights
    (standard deviation 0.01) and zero biases.
    """
    old_head = model.classifier[1]
    model.classifier[0] = nn.Dropout(p=dropout)
    if isinstance(old_head, nn.Linear):          # MobileNetV2
        new_head = nn.Linear(old_head.in_features, num_classes)
    else:                                        # SqueezeNet
        new_head = nn.Conv2d(old_head.in_channels, num_classes, kernel_size=1)
    nn.init.normal_(new_head.weight, mean=0.0, std=0.01)
    nn.init.zeros_(new_head.bias)
    model.classifier[1] = new_head
    model.num_classes = num_classes
    return model


def unfreeze_last_blocks(model, count):
    """Freeze every weight, then unfreeze the classifier and the last ``count`` feature blocks."""
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True
    for block in list(model.features.children())[-count:]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    return model


def build_mobilenet_v2(num_classes=NUM_CLASSES):
    settings = PRETRAINED_SETTINGS["mobilenet_v2"]
    model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    replace_classifier(model, num_classes, settings["dropout"])
    return unfreeze_last_blocks(model, settings["unfreeze_last_n"])


def build_squeezenet1_1(num_classes=NUM_CLASSES):
    settings = PRETRAINED_SETTINGS["squeezenet1_1"]
    model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1)
    replace_classifier(model, num_classes, settings["dropout"])
    return unfreeze_last_blocks(model, settings["unfreeze_last_n"])


MODEL_BUILDERS["mobilenet_v2"] = build_mobilenet_v2
MODEL_BUILDERS["squeezenet1_1"] = build_squeezenet1_1

## Counting multiply-accumulates

A multiply-accumulate (MAC) is one multiplication and one addition; one image's forward pass is
counted at batch size 1. Only convolutions and fully connected layers are counted, the convention
of the MobileNet and SqueezeNet papers and of torchvision's published figures (MobileNetV2 0.301 G
and SqueezeNet 1.1 0.349 G MACs at 224×224, which this count reproduces). Batch normalisation,
activations, pooling and bias additions are cheap next to the convolutions and are left out.

- a convolution costs (output values) × (input channels / groups) × (kernel area), so a depthwise
  convolution, with one group per channel, costs a fraction of a standard one;
- a fully connected layer costs (output values) × (input features).

In [6]:
import copy
import math

from torch import nn


def count_macs(model, input_shape=INPUT_SHAPE):
    """Multiply-accumulates of convolutions and fully connected layers, for one image.

    Measured on a copy in evaluation mode, so the model itself is untouched.
    """
    probe = copy.deepcopy(model).cpu().eval()
    total = 0

    def conv(module, inputs, output):
        nonlocal total
        total += output[0].numel() * (module.in_channels // module.groups) * math.prod(module.kernel_size)

    def linear(module, inputs, output):
        nonlocal total
        total += output[0].numel() * module.in_features

    hooks = []
    for module in probe.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(conv))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear))
    try:
        with torch.no_grad():
            probe(torch.zeros(1, *input_shape))
    finally:
        for hook in hooks:
            hook.remove()
    return int(total)

## Measuring what each model costs

None of these numbers depend on trained weights: an architecture costs the same to store and to run
whatever it has learned. So every model is built fresh, exactly as for training, and measured here,
all in one session on one machine.

- **Parameters**: trainable and total. Frozen pretrained weights still have to be stored.
- **Size on disk**: the saved weights (`state_dict`), as a device would store them, in KB
  (1 KB = 1,024 bytes). Saving them, rather than counting 4 bytes per parameter, includes buffers
  such as batch-normalisation statistics.
- **MACs** for one image, as counted above.
- **Latency**: milliseconds per image on **one CPU thread** at batch size 1, as a small edge device
  without a GPU would run it. 10 untimed passes first, then the mean of 100 timed passes; the median
  is recorded too.
- **Peak memory**: the weights plus the largest set of intermediate results alive at the same time
  during one forward pass, in MB (1 MB = 1,048,576 bytes). It is worked out from the model's graph
  (`torch.fx`): each intermediate result is freed after the last operation that uses it, and results
  that share memory (views, in-place operations) are counted once. It is therefore the same on any
  machine.

In [7]:
import copy
import itertools
import statistics
import time

from torch import fx

LATENCY_THREADS = 1
LATENCY_WARMUP = 10
LATENCY_REPEATS = 100


def latency_samples_ms(model, input_shape=INPUT_SHAPE, warmup=LATENCY_WARMUP, repeats=LATENCY_REPEATS):
    """Milliseconds of each timed forward pass of one image, on one CPU thread.

    The untimed passes first pay one-off costs, such as allocating memory, that
    would otherwise inflate the result. The thread setting is restored afterwards.
    """
    probe = copy.deepcopy(model).cpu().eval()
    batch = torch.randn(1, *input_shape, generator=torch.Generator().manual_seed(0))
    threads = torch.get_num_threads()
    torch.set_num_threads(LATENCY_THREADS)
    try:
        with torch.inference_mode():
            for _ in range(warmup):
                probe(batch)
            samples = []
            for _ in range(repeats):
                started = time.perf_counter()
                probe(batch)
                samples.append((time.perf_counter() - started) * 1000)
    finally:
        torch.set_num_threads(threads)
    return samples


def tensors_in(value):
    """Every tensor inside a value, however nested in tuples, lists or dicts."""
    if isinstance(value, torch.Tensor):
        yield value
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from tensors_in(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from tensors_in(item)


def weight_bytes(model):
    """Bytes of the parameters and buffers, each memory block counted once."""
    storages = {t.untyped_storage().data_ptr(): t.untyped_storage().nbytes()
                for t in itertools.chain(model.parameters(), model.buffers())}
    return sum(storages.values())


def activation_timeline(model, input_shape=INPUT_SHAPE):
    """Bytes of intermediate results alive while each step of the model's graph runs.

    Every value is kept until the pass ends, so no memory address is reused: that
    is what lets results sharing memory (views, in-place operations) be recognised
    by their address. A result counts from the step that creates it to the last
    step that uses it.
    """
    graph_module = fx.symbolic_trace(model)
    nodes = list(graph_module.graph.nodes)
    position = {node: index for index, node in enumerate(nodes)}
    values = {}

    class Recorder(fx.Interpreter):
        def run_node(self, node):
            value = super().run_node(node)
            values[node] = value
            return value

    with torch.no_grad():
        Recorder(graph_module, garbage_collect_values=False).run(torch.zeros(1, *input_shape))

    weights = {t.untyped_storage().data_ptr()
               for t in itertools.chain(graph_module.parameters(), graph_module.buffers())}
    storages = {}  # memory address -> [bytes, first step, last step]
    end = len(nodes) - 1
    for node in nodes:
        if node.op == "get_attr":  # weights and constants, counted with the weights
            continue
        last = end if node.op == "output" else max(
            (position[user] for user in node.users), default=position[node])
        for tensor in tensors_in(values.get(node)):
            storage = tensor.untyped_storage()
            address = storage.data_ptr()
            if address in weights:
                continue
            if address in storages:  # a view or an in-place result: the same memory
                storages[address][2] = max(storages[address][2], last)
            else:
                storages[address] = [storage.nbytes(), position[node], last]
    return [sum(size for size, first, final in storages.values() if first <= step <= final)
            for step in range(len(nodes))]


def peak_memory_mb(model, input_shape=INPUT_SHAPE):
    """Weights plus the peak of live intermediate results, in MB."""
    probe = copy.deepcopy(model).cpu().eval()
    return (weight_bytes(probe) + max(activation_timeline(probe, input_shape))) / 2**20


def profile_model(name):
    """Every cost of the model ``name``, built fresh as for training."""
    model = build_model(name)
    samples = latency_samples_ms(model)
    return {
        "model": name,
        "trainable_params": count_parameters(model),
        "total_params": count_parameters(model, trainable_only=False),
        "model_size_kb": model_size_kb(model),
        "macs": count_macs(model),
        "latency_ms": statistics.fmean(samples),
        "latency_median_ms": statistics.median(samples),
        "peak_memory_mb": peak_memory_mb(model),
        "device": describe_device("cpu"),
        "latency_threads": LATENCY_THREADS,
        "latency_repeats": LATENCY_REPEATS,
    }

## The machine

In [8]:
print("Latency is measured on:", describe_device("cpu"))

Latency is measured on: cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)


## Measure every model

In [9]:
benchmark = {}
for name in ("model_a", "model_b", "mobilenet_v2", "squeezenet1_1"):
    benchmark[name] = profile_model(name)
write_json(RESULTS_DIR / "benchmark.json", benchmark)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to data\torch_hub\checkpoints\mobilenet_v2-b0353104.pth


  0%|          | 0.00/13.6M [00:00<?, ?B/s]

  1%|          | 128k/13.6M [00:00<01:01, 229kB/s]

  2%|▏         | 256k/13.6M [00:00<00:34, 407kB/s]

  3%|▎         | 384k/13.6M [00:00<00:27, 503kB/s]

...

 90%|████████▉ | 4.25M/4.73M [00:06<00:00, 720kB/s]

 93%|█████████▎| 4.38M/4.73M [00:06<00:00, 714kB/s]

 98%|█████████▊| 4.62M/4.73M [00:06<00:00, 930kB/s]

100%|██████████| 4.73M/4.73M [00:06<00:00, 839kB/s]

100%|██████████| 4.73M/4.73M [00:06<00:00, 724kB/s]

## Results

In [10]:
from IPython.display import Markdown, display

columns = [  # (key, heading, format)
    ("trainable_params", "Trainable parameters", None),
    ("total_params", "Total parameters", None),
    ("model_size_kb", "Size (KB)", ",.1f"),
    ("macs", "MACs", None),
    ("latency_ms", "Latency (ms)", ".2f"),
    ("latency_median_ms", "Median latency (ms)", ".2f"),
    ("peak_memory_mb", "Peak memory (MB)", ".2f"),
]
display(Markdown(markdown_table(
    ["Model"] + [heading for _, heading, _ in columns],
    [[MODEL_LABELS[name]] + [costs[key] for key, _, _ in columns] for name, costs in benchmark.items()],
    caption=f"One image at a time, latency on one CPU thread of {benchmark['model_b']['device']}.",
    formats=[None] + [spec for _, _, spec in columns],
)))

*One image at a time, latency on one CPU thread of cpu (11th Gen Intel Core i5-11400H @ 2.70GHz).*

| Model | Trainable parameters | Total parameters | Size (KB) | MACs | Latency (ms) | Median latency (ms) | Peak memory (MB) |
|---|---:|---:|---:|---:|---:|---:|---:|
| Model A | 1,143,466 | 1,143,466 | 4,475.2 | 42,337,536 | 3.30 | 3.25 | 5.36 |
| Model B | 71,818 | 71,818 | 303.9 | 12,989,568 | 3.40 | 3.30 | 1.28 |
| MobileNetV2 | 1,218,890 | 2,236,682 | 8,971.1 | 24,461,312 | 6.22 | 5.82 | 9.41 |
| SqueezeNet 1.1 | 391,306 | 727,626 | 2,860.7 | 17,022,656 | 3.29 | 2.93 | 3.07 |
